                                    XGBoost Ranker Training for Recommendation System
The XGBoost ranker effectively learns to rank candidate items using behavioral, contextual, and embedding-based features. This enables the recommendation system to deliver highly relevant, personalized, and accurate item suggestions.
## Objective
To train a machine learning ranking model that predicts the relevance score of candidate items and ranks them accordingly for personalized recommendations.

## Data Source
The model uses the previously generated dataset:

ranker_training_dataset.csv → contains features, labels, and group IDs

## Data Loading and Inspection
The dataset is loaded into a DataFrame
Column names are cleaned
Data shape and sample rows are displayed
Purpose:
To verify dataset integrity before training.

## Data Validation

Ensures all required columns are present
Checks label distribution (positive vs negative samples)
Validates group IDs
Purpose:
To ensure the dataset is suitable for ranking model training.
## Data Cleaning and Preprocessing

Missing categorical values are replaced with “Unknown”
Numerical columns are converted to proper numeric types
Rows with invalid values are removed

## Feature Selection
Selected features include:

basket_size
item_cooccurrence_score
category_affinity_score
context_popularity_score
customer_history_score
embedding_similarity_score
isHoliday, isFestival, weekOfMonth
season, timeSlot, candidate_category

## Target:
label
Group:group_id
Purpose:
To define inputs for the ranking model.

## Feature Encoding
Categorical features are converted using one-hot encoding
Validation and test sets are aligned with training columns

## Data Sorting by Group
Data is sorted by group ID
Ensures that ranking model processes grouped data correctly

## Model Training (XGBoost Ranker)
The model is trained using:
Objective: rank
Learning rate, depth, and regularization parameters
LambdaRank strategy for pairwise ranking

## Sample Prediction Output
Displays ranked candidates for a sample group
Shows predicted scores and actual labels
## Benefits

Learns optimal ranking from multiple signals
Improves recommendation accuracy
Supports personalized and context-aware recommendations
Scalable and efficient for large datasets
Provides interpretable feature importance

## Recommendation Flow
Generate candidate items
Compute feature vectors
Apply trained XGBoost ranker
Sort items by predicted score
Recommend top-N items

In [1]:
!pip install xgboost


Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 1.0/101.7 MB 6.9 MB/s eta 0:00:15
   - -------------------------------------- 3.4/101.7 MB 9.2 MB/s eta 0:00:11
   - -------------------------------------- 4.5/101.7 MB 7.8 MB/s eta 0:00:13
   - -------------------------------------- 4.7/101.7 MB 6.0 MB/s eta 0:00:17
   -- ------------------------------------- 5.5/101.7 MB 5.4 MB/s eta 0:00:18
   -- ------------------------------------- 6.8/101.7 MB 5.5 MB/s eta 0:00:18
   --- ------------------------------------ 8.4/101.7 MB 5.8 MB/s eta 0:00:17
   --- ------------------------------------ 9.7/101.7 MB 5.9 MB/s eta 0:00:16
   ---- ----------------------------------- 11.3/101.7 MB 6.0 MB/s eta 0:00:16
   ----- ---------------------------------- 12.8/101.7 MB 6.2 MB/s eta 0:00:1

In [3]:
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd

import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import ndcg_score

In [4]:
DATA_DIR = Path(r"D:\recommendation_item_API\data")

RANKER_FILE = DATA_DIR / "ranker_training_dataset.csv"
MODEL_DIR = DATA_DIR / "model_outputs"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Ranker file exists:", RANKER_FILE.exists())
print("Model dir:", MODEL_DIR)

Ranker file exists: True
Model dir: C:\D drive\item_recommendation\data\model_outputs


In [5]:
ranker_df = pd.read_csv(RANKER_FILE)

ranker_df.columns = [c.strip() for c in ranker_df.columns]

print("Shape:", ranker_df.shape)
print("Columns:")
print(ranker_df.columns.tolist())

display(ranker_df.head())

Shape: (449680, 16)
Columns:
['customerId', 'candidate_item_id', 'candidate_category', 'basket_size', 'item_cooccurrence_score', 'category_affinity_score', 'context_popularity_score', 'customer_history_score', 'embedding_similarity_score', 'season', 'isHoliday', 'isFestival', 'timeSlot', 'weekOfMonth', 'label', 'group_id']


,customerId,candidate_item_id,candidate_category,basket_size,item_cooccurrence_score,category_affinity_score,context_popularity_score,customer_history_score,embedding_similarity_score,season,isHoliday,isFestival,timeSlot,weekOfMonth,label,group_id
0,23561,7075,Dairy-Other,2,6.0,2.0,4.0,2.0,0.992026,Winter,0,0,Morning,1,1,1
1,23561,952,Pantry-Rice,2,26.0,3.0,4.0,0.0,0.979893,Winter,0,0,Morning,1,0,1
2,23561,26615,Instant-Food,2,0.0,5.0,1.0,3.0,0.997745,Winter,0,0,Morning,1,0,1
3,23561,14034,Protein-Egg,2,29.0,4.0,7.0,2.0,0.998886,Winter,0,0,Morning,1,0,1
4,23561,2030,Pantry-Rice,2,23.5,3.0,2.0,3.0,0.979893,Winter,0,0,Morning,1,0,1


In [6]:
ranker_df = pd.read_csv(RANKER_FILE)

ranker_df.columns = [c.strip() for c in ranker_df.columns]

print("Shape:", ranker_df.shape)
print("Columns:")
print(ranker_df.columns.tolist())

display(ranker_df.head())

Shape: (449680, 16)
Columns:
['customerId', 'candidate_item_id', 'candidate_category', 'basket_size', 'item_cooccurrence_score', 'category_affinity_score', 'context_popularity_score', 'customer_history_score', 'embedding_similarity_score', 'season', 'isHoliday', 'isFestival', 'timeSlot', 'weekOfMonth', 'label', 'group_id']


,customerId,candidate_item_id,candidate_category,basket_size,item_cooccurrence_score,category_affinity_score,context_popularity_score,customer_history_score,embedding_similarity_score,season,isHoliday,isFestival,timeSlot,weekOfMonth,label,group_id
0,23561,7075,Dairy-Other,2,6.0,2.0,4.0,2.0,0.992026,Winter,0,0,Morning,1,1,1
1,23561,952,Pantry-Rice,2,26.0,3.0,4.0,0.0,0.979893,Winter,0,0,Morning,1,0,1
2,23561,26615,Instant-Food,2,0.0,5.0,1.0,3.0,0.997745,Winter,0,0,Morning,1,0,1
3,23561,14034,Protein-Egg,2,29.0,4.0,7.0,2.0,0.998886,Winter,0,0,Morning,1,0,1
4,23561,2030,Pantry-Rice,2,23.5,3.0,2.0,3.0,0.979893,Winter,0,0,Morning,1,0,1


In [7]:
required_cols = [
    "group_id",
    "label",
    "candidate_item_id",
    "candidate_category",
    "basket_size",
    "item_cooccurrence_score",
    "category_affinity_score",
    "context_popularity_score",
    "customer_history_score",
    "embedding_similarity_score",
    "season",
    "isHoliday",
    "isFestival",
    "timeSlot",
    "weekOfMonth"
]

missing_cols = [c for c in required_cols if c not in ranker_df.columns]
if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

print("Unique groups:", ranker_df["group_id"].nunique())
print("Label distribution:")
print(ranker_df["label"].value_counts(dropna=False))

Unique groups: 40880
Label distribution:
label
0    408800
1     40880
Name: count, dtype: int64


In [8]:
ranker_df["candidate_category"] = ranker_df["candidate_category"].fillna("Unknown").astype(str)
ranker_df["season"] = ranker_df["season"].fillna("Unknown").astype(str)
ranker_df["timeSlot"] = ranker_df["timeSlot"].fillna("Unknown").astype(str)

numeric_cols = [
    "basket_size",
    "item_cooccurrence_score",
    "category_affinity_score",
    "context_popularity_score",
    "customer_history_score",
    "embedding_similarity_score",
    "isHoliday",
    "isFestival",
    "weekOfMonth",
    "label"
]

for col in numeric_cols:
    ranker_df[col] = pd.to_numeric(ranker_df[col], errors="coerce")

ranker_df = ranker_df.dropna(subset=numeric_cols).copy()

print("Shape after cleaning:", ranker_df.shape)
display(ranker_df.head())

Shape after cleaning: (449680, 16)


,customerId,candidate_item_id,candidate_category,basket_size,item_cooccurrence_score,category_affinity_score,context_popularity_score,customer_history_score,embedding_similarity_score,season,isHoliday,isFestival,timeSlot,weekOfMonth,label,group_id
0,23561,7075,Dairy-Other,2,6.0,2.0,4.0,2.0,0.992026,Winter,0,0,Morning,1,1,1
1,23561,952,Pantry-Rice,2,26.0,3.0,4.0,0.0,0.979893,Winter,0,0,Morning,1,0,1
2,23561,26615,Instant-Food,2,0.0,5.0,1.0,3.0,0.997745,Winter,0,0,Morning,1,0,1
3,23561,14034,Protein-Egg,2,29.0,4.0,7.0,2.0,0.998886,Winter,0,0,Morning,1,0,1
4,23561,2030,Pantry-Rice,2,23.5,3.0,2.0,3.0,0.979893,Winter,0,0,Morning,1,0,1


In [9]:
all_groups = ranker_df["group_id"].drop_duplicates().tolist()
random.seed(42)
random.shuffle(all_groups)

n_groups = len(all_groups)
train_end = int(n_groups * 0.80)
valid_end = int(n_groups * 0.90)

train_groups = set(all_groups[:train_end])
valid_groups = set(all_groups[train_end:valid_end])
test_groups  = set(all_groups[valid_end:])

train_df = ranker_df[ranker_df["group_id"].isin(train_groups)].copy()
valid_df = ranker_df[ranker_df["group_id"].isin(valid_groups)].copy()
test_df  = ranker_df[ranker_df["group_id"].isin(test_groups)].copy()

print("Train shape:", train_df.shape, "groups:", train_df["group_id"].nunique())
print("Valid shape:", valid_df.shape, "groups:", valid_df["group_id"].nunique())
print("Test shape :", test_df.shape,  "groups:", test_df["group_id"].nunique())

Train shape: (359744, 16) groups: 32704
Valid shape: (44968, 16) groups: 4088
Test shape : (44968, 16) groups: 4088


In [10]:
feature_cols = [
    "basket_size",
    "item_cooccurrence_score",
    "category_affinity_score",
    "context_popularity_score",
    "customer_history_score",
    "embedding_similarity_score",
    "isHoliday",
    "isFestival",
    "weekOfMonth",
    "season",
    "timeSlot",
    "candidate_category"
]

target_col = "label"
group_col = "group_id"

print("Feature columns:", feature_cols)

Feature columns: ['basket_size', 'item_cooccurrence_score', 'category_affinity_score', 'context_popularity_score', 'customer_history_score', 'embedding_similarity_score', 'isHoliday', 'isFestival', 'weekOfMonth', 'season', 'timeSlot', 'candidate_category']


In [11]:
X_train_raw = train_df[feature_cols].copy()
X_valid_raw = valid_df[feature_cols].copy()
X_test_raw  = test_df[feature_cols].copy()

X_train = pd.get_dummies(X_train_raw, columns=["season", "timeSlot", "candidate_category"])
X_valid = pd.get_dummies(X_valid_raw, columns=["season", "timeSlot", "candidate_category"])
X_test  = pd.get_dummies(X_test_raw,  columns=["season", "timeSlot", "candidate_category"])

# train columns অনুযায়ী align
X_valid = X_valid.reindex(columns=X_train.columns, fill_value=0)
X_test  = X_test.reindex(columns=X_train.columns, fill_value=0)

y_train = train_df[target_col].astype(int).values
y_valid = valid_df[target_col].astype(int).values
y_test  = test_df[target_col].astype(int).values

qid_train = train_df[group_col].values
qid_valid = valid_df[group_col].values
qid_test  = test_df[group_col].values

print("X_train shape:", X_train.shape)
print("X_valid shape:", X_valid.shape)
print("X_test shape :", X_test.shape)

X_train shape: (359744, 59)
X_valid shape: (44968, 59)
X_test shape : (44968, 59)


In [12]:
def sort_by_qid(X, y, qid):
    temp = X.copy()
    temp["_label"] = y
    temp["_qid"] = qid
    
    temp = temp.sort_values("_qid").reset_index(drop=True)
    
    y_sorted = temp["_label"].values
    qid_sorted = temp["_qid"].values
    X_sorted = temp.drop(columns=["_label", "_qid"])
    
    return X_sorted, y_sorted, qid_sorted

X_train_sorted, y_train_sorted, qid_train_sorted = sort_by_qid(X_train, y_train, qid_train)
X_valid_sorted, y_valid_sorted, qid_valid_sorted = sort_by_qid(X_valid, y_valid, qid_valid)
X_test_sorted, y_test_sorted, qid_test_sorted = sort_by_qid(X_test, y_test, qid_test)

print("Sorted train shape:", X_train_sorted.shape)

Sorted train shape: (359744, 59)


In [13]:
ranker = xgb.XGBRanker(
    objective="rank:ndcg",
    tree_method="hist",
    learning_rate=0.05,
    max_depth=6,
    n_estimators=300,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    random_state=42,
    lambdarank_pair_method="topk",
    lambdarank_num_pair_per_sample=8
)

ranker.fit(
    X_train_sorted,
    y_train_sorted,
    qid=qid_train_sorted,
    eval_set=[(X_valid_sorted, y_valid_sorted)],
    eval_qid=[qid_valid_sorted],
    verbose=True
)

[0]	validation_0-ndcg@8:0.53115
[1]	validation_0-ndcg@8:0.57594
[2]	validation_0-ndcg@8:0.60644
[3]	validation_0-ndcg@8:0.62782
[4]	validation_0-ndcg@8:0.64200
[5]	validation_0-ndcg@8:0.64720
[6]	validation_0-ndcg@8:0.65123
[7]	validation_0-ndcg@8:0.65295
[8]	validation_0-ndcg@8:0.65432
[9]	validation_0-ndcg@8:0.65544
[10]	validation_0-ndcg@8:0.65477
[11]	validation_0-ndcg@8:0.65395
[12]	validation_0-ndcg@8:0.65464
[13]	validation_0-ndcg@8:0.65611
[14]	validation_0-ndcg@8:0.65596
[15]	validation_0-ndcg@8:0.65575
[16]	validation_0-ndcg@8:0.65516
[17]	validation_0-ndcg@8:0.65663
[18]	validation_0-ndcg@8:0.65634
[19]	validation_0-ndcg@8:0.65831
[20]	validation_0-ndcg@8:0.65692
[21]	validation_0-ndcg@8:0.65645
[22]	validation_0-ndcg@8:0.65692
[23]	validation_0-ndcg@8:0.65668
[24]	validation_0-ndcg@8:0.65895
[25]	validation_0-ndcg@8:0.66185
[26]	validation_0-ndcg@8:0.66329
[27]	validation_0-ndcg@8:0.66179
[28]	validation_0-ndcg@8:0.66225
[29]	validation_0-ndcg@8:0.66248
[30]	validation_0-nd

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'rank:ndcg'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sk

In [14]:
train_df_sorted = train_df.copy()
valid_df_sorted = valid_df.copy()
test_df_sorted = test_df.copy()

# একই sorting apply
train_df_sorted = train_df_sorted.assign(_row_order=np.arange(len(train_df_sorted)))
valid_df_sorted = valid_df_sorted.assign(_row_order=np.arange(len(valid_df_sorted)))
test_df_sorted  = test_df_sorted.assign(_row_order=np.arange(len(test_df_sorted)))

train_df_sorted = train_df_sorted.sort_values(group_col).reset_index(drop=True)
valid_df_sorted = valid_df_sorted.sort_values(group_col).reset_index(drop=True)
test_df_sorted  = test_df_sorted.sort_values(group_col).reset_index(drop=True)

train_scores = ranker.predict(X_train_sorted)
valid_scores = ranker.predict(X_valid_sorted)
test_scores  = ranker.predict(X_test_sorted)

train_df_sorted["pred_score"] = train_scores
valid_df_sorted["pred_score"] = valid_scores
test_df_sorted["pred_score"] = test_scores

display(test_df_sorted.head())

,customerId,candidate_item_id,candidate_category,basket_size,item_cooccurrence_score,category_affinity_score,context_popularity_score,customer_history_score,embedding_similarity_score,season,isHoliday,isFestival,timeSlot,weekOfMonth,label,group_id,_row_order,pred_score
0,23494,708,Spices-Cooking,6,14.166667,94.0,2.0,3.0,0.979264,Winter,0,0,Afternoon,1,1,27,0,0.107744
1,23494,13352,Protein-Egg,6,22.000000,42.0,3.0,25.0,0.997620,Winter,0,0,Afternoon,1,0,27,1,-0.099650
2,23494,6815,Beverage-Hot,6,5.666667,16.0,4.0,2.0,0.998512,Winter,0,0,Afternoon,1,0,27,2,-0.238307
3,23494,15279,Veg-Cooking,6,6.833333,47.0,3.0,5.0,0.987591,Winter,0,0,Afternoon,1,0,27,3,0.728479
4,23494,13958,Fish-Fresh,6,14.500000,106.0,2.0,31.0,0.988419,Winter,0,0,Afternoon,1,0,27,4,0.013488


In [15]:
def precision_at_k(y_true, y_score, k=5):
    order = np.argsort(y_score)[::-1][:k]
    topk_true = np.array(y_true)[order]
    return float(np.sum(topk_true)) / float(k)


def recall_at_k(y_true, y_score, k=5):
    y_true = np.array(y_true)
    total_relevant = np.sum(y_true)
    
    if total_relevant == 0:
        return 0.0
    
    order = np.argsort(y_score)[::-1][:k]
    topk_true = y_true[order]
    return float(np.sum(topk_true)) / float(total_relevant)


def ndcg_at_k(y_true, y_score, k=5):
    y_true = np.array(y_true).reshape(1, -1)
    y_score = np.array(y_score).reshape(1, -1)
    return float(ndcg_score(y_true, y_score, k=k))

In [16]:
def evaluate_grouped_ranking(df_scored, group_col="group_id", label_col="label", score_col="pred_score", k=5):
    precision_scores = []
    recall_scores = []
    ndcg_scores = []
    
    for gid, grp in df_scored.groupby(group_col):
        y_true = grp[label_col].values
        y_score = grp[score_col].values
        
        precision_scores.append(precision_at_k(y_true, y_score, k=k))
        recall_scores.append(recall_at_k(y_true, y_score, k=k))
        ndcg_scores.append(ndcg_at_k(y_true, y_score, k=k))
    
    return {
        f"Precision@{k}": float(np.mean(precision_scores)),
        f"Recall@{k}": float(np.mean(recall_scores)),
        f"NDCG@{k}": float(np.mean(ndcg_scores))
    }

train_metrics_k5 = evaluate_grouped_ranking(train_df_sorted, k=5)
valid_metrics_k5 = evaluate_grouped_ranking(valid_df_sorted, k=5)
test_metrics_k5  = evaluate_grouped_ranking(test_df_sorted, k=5)

print("Train metrics @5:", train_metrics_k5)
print("Valid metrics @5:", valid_metrics_k5)
print("Test metrics @5 :", test_metrics_k5)

Train metrics @5: {'Precision@5': 0.1841059197651664, 'Recall@5': 0.9205295988258317, 'NDCG@5': 0.7303562272985135}
Valid metrics @5: {'Precision@5': 0.18160469667318985, 'Recall@5': 0.9080234833659491, 'NDCG@5': 0.7018862283111399}
Test metrics @5 : {'Precision@5': 0.1802837573385519, 'Recall@5': 0.9014187866927593, 'NDCG@5': 0.7042181267229491}


In [17]:
for k in [3, 5, 10]:
    print(f"\n===== Metrics at K = {k} =====")
    print("Train:", evaluate_grouped_ranking(train_df_sorted, k=k))
    print("Valid:", evaluate_grouped_ranking(valid_df_sorted, k=k))
    print("Test :", evaluate_grouped_ranking(test_df_sorted, k=k))


===== Metrics at K = 3 =====
Train: {'Precision@3': 0.26660347358121333, 'Recall@3': 0.79981042074364, 'NDCG@3': 0.6805241820380803}
Valid: {'Precision@3': 0.25627853881278534, 'Recall@3': 0.7688356164383562, 'NDCG@3': 0.6444854500161674}
Test : {'Precision@3': 0.25962165688193084, 'Recall@3': 0.7788649706457925, 'NDCG@3': 0.6536454107439945}

===== Metrics at K = 5 =====
Train: {'Precision@5': 0.1841059197651664, 'Recall@5': 0.9205295988258317, 'NDCG@5': 0.7303562272985135}
Valid: {'Precision@5': 0.18160469667318985, 'Recall@5': 0.9080234833659491, 'NDCG@5': 0.7018862283111399}
Test : {'Precision@5': 0.1802837573385519, 'Recall@5': 0.9014187866927593, 'NDCG@5': 0.7042181267229491}

===== Metrics at K = 10 =====
Train: {'Precision@10': 0.09985934442270063, 'Recall@10': 0.9985934442270059, 'NDCG@10': 0.7564392458043019}
Valid: {'Precision@10': 0.09980430528375736, 'Recall@10': 0.9980430528375733, 'NDCG@10': 0.7320836140516208}
Test : {'Precision@10': 0.09985322896281802, 'Recall@10': 0

In [18]:
feature_importance_df = pd.DataFrame({
    "feature": X_train_sorted.columns,
    "importance": ranker.feature_importances_
}).sort_values("importance", ascending=False)

display(feature_importance_df.head(20))

,feature,importance
48,candidate_category_Personal-Care-Bath,0.047126
49,candidate_category_Personal-Care-Cosmetics,0.046810
38,candidate_category_Meat-Fresh,0.045158
50,candidate_category_Personal-Care-Hair,0.037961
30,candidate_category_Fruits-Fresh,0.033587
37,candidate_category_Instant-Food,0.033158
26,candidate_category_Dairy-Other,0.031647
44,candidate_category_Pantry-Oils,0.031475
51,candidate_category_Personal-Care-Oral,0.029536
58,candidate_category_Veg-Cooking,0.028652


In [19]:
MODEL_FILE = MODEL_DIR / "xgboost_ranker_model.json"
FEATURE_FILE = MODEL_DIR / "ranker_feature_columns.json"
IMPORTANCE_FILE = MODEL_DIR / "ranker_feature_importance.csv"

ranker.save_model(MODEL_FILE)

with open(FEATURE_FILE, "w", encoding="utf-8") as f:
    json.dump(list(X_train_sorted.columns), f, ensure_ascii=False, indent=2)

feature_importance_df.to_csv(IMPORTANCE_FILE, index=False)

print("Saved model:", MODEL_FILE)
print("Saved feature list:", FEATURE_FILE)
print("Saved importance:", IMPORTANCE_FILE)

Saved model: C:\D drive\item_recommendation\data\model_outputs\xgboost_ranker_model.json
Saved feature list: C:\D drive\item_recommendation\data\model_outputs\ranker_feature_columns.json
Saved importance: C:\D drive\item_recommendation\data\model_outputs\ranker_feature_importance.csv


In [20]:
sample_group = test_df_sorted[group_col].iloc[0]
sample_view = test_df_sorted[test_df_sorted[group_col] == sample_group].copy()

sample_view = sample_view.sort_values("pred_score", ascending=False)

display(
    sample_view[
        [
            "group_id",
            "candidate_item_id",
            "candidate_category",
            "label",
            "pred_score",
            "basket_size",
            "item_cooccurrence_score",
            "category_affinity_score",
            "context_popularity_score",
            "customer_history_score",
            "embedding_similarity_score"
        ]
    ]
)

,group_id,candidate_item_id,candidate_category,label,pred_score,basket_size,item_cooccurrence_score,category_affinity_score,context_popularity_score,customer_history_score,embedding_similarity_score
3,27,15279,Veg-Cooking,0,0.728479,6,6.833333,47.0,3.0,5.0,0.987591
5,27,15104,Veg-Cooking,0,0.642522,6,7.166667,47.0,3.0,2.0,0.987591
10,27,952,Pantry-Rice,0,0.153336,6,73.833333,67.0,6.0,25.0,0.977824
0,27,708,Spices-Cooking,1,0.107744,6,14.166667,94.0,2.0,3.0,0.979264
7,27,332,Spices-Cooking,0,0.067833,6,17.333333,94.0,3.0,12.0,0.979264
4,27,13958,Fish-Fresh,0,0.013488,6,14.500000,106.0,2.0,31.0,0.988419
1,27,13352,Protein-Egg,0,-0.099650,6,22.000000,42.0,3.0,25.0,0.997620
2,27,6815,Beverage-Hot,0,-0.238307,6,5.666667,16.0,4.0,2.0,0.998512
6,27,31928,Clothing-Accessories,0,-0.510198,6,12.500000,13.0,3.0,3.0,0.995363
8,27,13988,Meat-Fresh,0,-0.827429,6,28.000000,99.0,1.0,4.0,0.988786
